## agent를 만들어야 하는 이유 

In [4]:
from dotenv import load_dotenv
from langchain_openai.chat_models.base import ChatOpenAI
import os

load_dotenv()
print(os.environ.get('OPENAI_API_KEY')[:20])

sk-proj-ktAbTJ1mXZ2V


In [5]:
# 5,243.38
prompt = "cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $522.00 + $76.16 + $29.12"

In [6]:
chat = ChatOpenAI(temperature = 0.1 )

In [7]:
result = chat.invoke(prompt)

In [8]:
result.content

'The total cost is $4,363.38.'

In [9]:
chat = ChatOpenAI(model="gpt-4o", temperature = 0.1 )

In [10]:
result = chat.invoke(prompt)

In [11]:
result.content

'To find the total cost, you need to add all the amounts together:\n\n$355.39 + $924.87 + $721.20 + $1940.29 + $573.63 + $65.72 + $35.00 + $522.00 + $76.16 + $29.12 = $5243.38\n\nThe total cost is $5243.38.'

In [12]:
"""
    프롬프트의 정답
    $4,363.38.

    계산기에서 직접 계산하기 ↓↓↓
    $5,243.38

    llm의 계산 착오
    ※이유※
    LLM은 산술 연산을 수행하지 않습니다. 이런 계산은 AI보다 계산기가 더 잘합니다.
    LLM은 text를 생성해내는 모델입니다. 문장의 시퀀스의 다음 token이 무엇인지 통계적으로 추측합니다.
    이러한 LLm의 오류를 잡기 위해서는 agent를 제공해주어야 합니다.
    그리고 agent를 위한 tool(툴)을 만들고, agent가 tool을 선택해서 실행하는 것입니다.
"""

'\n    프롬프트의 정답\n    $4,363.38.\n\n    계산기에서 직접 계산하기 ↓↓↓\n    $5,243.38\n\n    llm의 계산 착오\n    ※이유※\n    LLM은 산술 연산을 수행하지 않습니다. 이런 계산은 AI보다 계산기가 더 잘합니다.\n    LLM은 text를 생성해내는 모델입니다. 문장의 시퀀스의 다음 token이 무엇인지 통계적으로 추측합니다.\n    이러한 LLm의 오류를 잡기 위해서는 agent를 제공해주어야 합니다.\n    그리고 agent를 위한 tool(툴)을 만들고, agent가 tool을 선택해서 실행하는 것입니다.\n'

## Agent 생성`

In [13]:
# create_agent : 함수로 통일 
from langchain.agents import create_agent
from langchain.tools import tool

In [14]:
def plus(num1,num2) :
    """
        Adds two numbers and return the result.
    """
    return num1 + num2
"""
   json {
        "name": plus,
        "description": "Adds two numbers and return the result."
       
   }
"""

'\n   json {\n        "name": plus,\n        "description": "Adds two numbers and return the result."\n\n   }\n'

In [15]:
agent = create_agent(
    model = "gpt-3.5-turbo",
    tools = [plus] ,
    system_prompt= "You are a helpful assistant"
)

In [16]:
result = agent.invoke({
    "messages" : [
        {
            "role":"user",
            "content": prompt
        }
    ]
})


In [17]:
result

{'messages': [HumanMessage(content='cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $522.00 + $76.16 + $29.12', additional_kwargs={}, response_metadata={}, id='6daf6b8b-b45e-4db8-b2e5-df54628457b2'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 131, 'prompt_tokens': 109, 'total_tokens': 240, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-Dh4kSe81Jb0keIAFJD3Rz3Jg57yOO', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e3e1f-bf44-7151-b4ec-15ff6ee26e75-0', tool_calls=[{'name': 'plus', 'args': {'num1': 355.39, 'num2': 924.87}, 'id': 'call_YVcretZyAOK2EZ4QDNANsgs5', 'type': 'tool_call'},

In [18]:
for message in result["messages"]:
    if message.__class__.__name__ == "AIMessage" and message.tool_calls:
        for i in message.tool_calls:
            print(i)

{'name': 'plus', 'args': {'num1': 355.39, 'num2': 924.87}, 'id': 'call_YVcretZyAOK2EZ4QDNANsgs5', 'type': 'tool_call'}
{'name': 'plus', 'args': {'num1': 721.2, 'num2': 1940.29}, 'id': 'call_Vk2PG0Fx2dY0wEKrtJk4322v', 'type': 'tool_call'}
{'name': 'plus', 'args': {'num1': 573.63, 'num2': 65.72}, 'id': 'call_WnAJeJcXPyVFeHkPxX9mSymM', 'type': 'tool_call'}
{'name': 'plus', 'args': {'num1': 35.0, 'num2': 522.0}, 'id': 'call_TJFqJoXRJPeTssDXrIgabkX3', 'type': 'tool_call'}
{'name': 'plus', 'args': {'num1': 76.16, 'num2': 29.12}, 'id': 'call_KPiMeg4X9OQfJ6K6s6qBANJ7', 'type': 'tool_call'}
{'name': 'plus', 'args': {'num1': 105.28, 'num2': 640.0}, 'id': 'call_Ft3XNNAMMmiJCKVLZsJ6dd6S', 'type': 'tool_call'}


In [19]:
@tool
def total_sum(numbers : list[float]) -> float : 
    """
        Adds a list of numbers and returns the total sum.
        Use this tool when you need to calculate the total of multiple numbers.
        Input should be a string representation of a list.
        Example: "[1, 2, 3]"
    """

    return sum(numbers)

In [20]:
agent = create_agent(
    model = "gpt-3.5-turbo",
    tools = [total_sum] ,
    system_prompt= "You are a helpful assistant"
)

In [21]:
result = agent.invoke({
    "messages" : [
        {
            "role":"user",
            "content": prompt
        }
    ]
})

In [27]:
for message in result["messages"]:
    if message.__class__.__name__ == "AIMessage" and message.tool_calls:
        for i in message.tool_calls:
            print(i)

{'name': 'total_sum', 'args': {'numbers': [355.39, 924.87, 721.2, 1940.29, 573.63, 65.72, 35.0, 522.0, 76.16, 29.12]}, 'id': 'call_PzI7al6P0TygciwwyHiyuSTw', 'type': 'tool_call'}


## 랭스미스ㅡ

# LangSmith(랭스미스)
- LLM 기반 애플리케이션의 디버깅, 성능 평가, 모니터링 등을 제공하는 랭체인의 통합 플랫폼입니다.

## 랭스미스 Open API Key 발급
- https://smith.langchain.com/ 접속
- 로그인 후 좌측 하단 [Setting] 메뉴 클릭
- [API Keys] 클릭 후 생성
- Description은 lang_ksh(이니셜)
- [default workspace]는 기존에 있는 workspace1로 만들고 생성
- 발급받은 KEY를 .env에 추가하기

### .env에 추가하기
- LANGCHAIN_TRACING_V2=true
- LANGCHAIN_ENDPOINT="https://api.smith.langchain.com"
- LANGCHAIN_PROJECT=lang_1900
- LANGSMITH_API_KEY=발급받은 랭스미스 key

### 설정 후 Jupyter Notebook 재실행

## Agent가 동작하는 과정
1. 끝날때까지 반복이 되는 loop입니다.
2. llm으로 부터 어떤 것을 할지(get action)을 받아온다. (lang smith의 output에서 확인가능)
3. 실행한 결과를 observation이라고 부른다. 다시 다음 next action을 실행시킨다.
4. Agent Finish를 응답받으면 마지막 action 값을 리턴한다.

## 1. ReAct (Reasoning and Acting)

In [27]:
from dotenv import load_dotenv
import os

from langchain_core.prompts import ChatPromptTemplate
from langchain.tools import tool, BaseTool
from langchain_classic.agents import AgentExecutor, create_react_agent, create_openai_functions_agent
from langchain_classic import hub
from langchain.agents import create_agent
from langchain_openai.chat_models.base import ChatOpenAI

from pydantic import BaseModel, Field
from typing import Any, Type, List #Python의 내장 모듈 typing

In [28]:
load_dotenv()

True

In [29]:
llm = ChatOpenAI(model = "gpt-4o-mini" , temperature = 0)

In [30]:
@tool
def plus(expression: str) -> float:
    """
        Adds multiple numbers and returns their total sum.

        The input must be a comma-spreadted string of numbers.
        Example: "10,20,30"

        Use this tool then when you nee to calcuate the sum of multiple values.
    """
    try:
        numbers = [float (num) for num in inputs.split(",")]
        return sum(numbers)
    except Exception as e:
        return -1



In [37]:
tools = [plus]

react_agent_prompt = hub.pull("hwchase17/react")

# 판단
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=react_agent_prompt
)

# 실행기
react_agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True  # 내부동작 확인
)

In [40]:
prompt = "cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $522.00 + $76.16 + $29.12"

react_agent_executor.invoke({
    "input": prompt
})



> Entering new AgentExecutor chain...
To find the total cost, I need to sum all the given amounts. I will use the plus tool to calculate the total.

Action: plus
Action Input: "355.39,924.87,721.2,1940.29,573.63,65.72,35.00,522.00,76.16,29.12"-1It seems there was an issue with the calculation. I will double-check the input format and try again to ensure that the numbers are correctly formatted for the plus tool.

Action: plus
Action Input: "355.39,924.87,721.2,1940.29,573.63,65.72,35.00,522.00,76.16,29.12"-1It appears that the plus tool is not accepting the input as expected. I will try to format the numbers differently or check for any potential issues with the input.

Action: plus
Action Input: "355.39, 924.87, 721.2, 1940.29, 573.63, 65.72, 35.00, 522.00, 76.16, 29.12"-1It seems that the plus tool is not accepting the input due to formatting issues. I will try removing any spaces and ensure that the numbers are formatted correctly.

Action: plus  
Action Input: "355.39,924.87,721.

{'input': 'cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $522.00 + $76.16 + $29.12',
 'output': '$4721.38'}

## 2. OpenAI Function Calling Agent

In [6]:
class CalculatorToolArgsSchema(BaseModel):
    numbers:List[float] = Field(description= "Numbers to sum")

class CalculatorTool(BaseTool):
    #약속된 필드 이름 (공백x, 한글x ,a-z,A-Z,0-9,_,- 만 가능)
    name : Type[str] = "calculator_tool"
    description: Type[str] = """
        Add multiple numbers and returns their total sum.
        ust this tools when you need to calculator the sum of nultiple values.
    """

    args_schema: Type[BaseModel] = CalculatorToolArgsSchema

    
    # BaseTool은 반드시 _run 함수를 재정의
    # tool을 호출 했을떄 실행되는 메인 로직

    def _run (self,numvers) :
        return sum(numbers)

    

NameError: name 'BaseModel' is not defined

In [5]:
tools = [CalculatorTool()]

# placeholder(agent_scratchpad): 내부 tool, reasoning 호출 기록을 임시 저장
function_agent_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("placeholder", """{agent_scratchpad}"""),
])

agent = create_openai_functions_agent(
    llm=llm,
    tools=tools,
    prompt=function_agent_prompt
)

#실행기
calling_agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True  # 내부동작 확인
)


NameError: name 'CalculatorTool' is not defined

In [2]:
prompt = "cost of $355.39 + $924.87 + $71.2 + $194.29 + $73.63 + $6.72 + $3.00 + $22.00 + $7616 + $2.12"


result= calling_agent_executor.invoke({
    "input": prompt
})

NameError: name 'calling_agent_executor' is not defined

In [3]:
result["output"]

NameError: name 'result' is not defined

# V1.0 ^ creat_agent(커스텀 툴)

In [14]:
from dotenv import load_dotenv
import os
import requests

from langchain_core.prompts import ChatPromptTemplate
from langchain.tools import tool, BaseTool
from langchain.agents import create_agent
from langchain_openai.chat_models.base import ChatOpenAI

from pydantic import BaseModel, Field
from typing import Any, Type, List ,Tuple , Dict #Python의 내장 모듈 typing

# 추가된 import
from langchain_community.utilities.duckduckgo_search import DuckDuckGoSearchAPIWrapper
from geopy.geocoders import Nominatim

load_dotenv()
print(os.environ.get('OPENAI_API_KEY')[:20])

sk-proj-ktAbTJ1mXZ2V


In [15]:
tools = []

agent = create_agent(
    model="gpt-4o-mini",
    tools=tools
)

In [16]:
prompt = ChatPromptTemplate.from_messages([
    ("human", "중고등학교, 대학교의 시험기간 끝나는 주를 알려줘")
])

chain = prompt | agent
result = chain.invoke({})

print(result)

{'messages': [HumanMessage(content='중고등학교, 대학교의 시험기간 끝나는 주를 알려줘', additional_kwargs={}, response_metadata={}, id='90d514b7-e61a-4bf6-b9a1-f74a9f7d5059'), AIMessage(content='중고등학교 및 대학교의 시험 기간은 국가, 교육 기관, 학과에 따라 다를 수 있습니다. 대한민국의 경우, 일반적으로 중고등학교는 1학기 중간고사가 5월 말에서 6월 초, 기말고사가 7월 중순에 실시됩니다. 2학기 중간고사는 10월 초, 기말고사는 12월 중순에 이루어집니다.\n\n대학교는 학사 및 전공에 따라 다르지만, 일반적으로 1학기 중간고사는 4월 말에서 5월 초, 기말고사는 6월 중순에 실시됩니다. 2학기 중간고사는 10월 초, 기말고사는 12월 중순에서 말에 걸쳐 이루어집니다.\n\n정확한 시험 일정은 해당 학교의 학사 일정에 따라 다르므로, 각 학교의 공식 웹사이트나 학사 일정을 참고하는 것이 좋습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 208, 'prompt_tokens': 23, 'total_tokens': 231, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_22a0c4db5d', 'id': 'chatcmpl-Dh8WdSfd

In [17]:
# 지역 -> 위도, 경도
def get_coordinates(location_name):
    locator = Nominatim(user_agent= "kyj")
    location = locator.geocode(location_name)

    print(location)
    print(location.longitude)
    print(location.latitude)

    return location.latitude, location.longitude,

In [18]:
get_coordinates("강남")

강남, 테헤란로, 역삼1동, 강남구, 서울특별시, 06134, 대한민국
127.0275574
37.4979497


(37.4979497, 127.0275574)

In [19]:
import requests

def get_weather(lat, lon):

    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"

    weather_codes = {
        0: "맑음 ☀️",
        1: "대체로 맑음 🌤️",
        2: "구름 조금 ⛅",
        3: "흐림 ☁️",
        45: "안개 🌫️",
        48: "침강 안개 🌫️",
        51: "가벼운 이슬비 🌦️",
        53: "이슬비 🌧️",
        55: "강한 이슬비 ⛈️",
        61: "약한 비 💧",
        63: "보통 비 ☔",
        65: "강한 비 🌊",
        71: "약한 눈 ❄️",
        73: "보통 눈 ☃️",
        75: "강한 눈 🏔️",
        80: "약한 소나기 🌦️",
        81: "보통 소나기 🌧️",
        82: "강한 소나기 ⛈️",
        95: "뇌우 ⚡",
        96: "뇌우 및 우박 ⛈️",
        99: "심한 뇌우 🌪️"
    }

    try:
        response = requests.get(url)

        data = response.json()

        if "current_weather" in data:

            current = data["current_weather"]

            temp = current["temperature"]
            wind = current["windspeed"]
            code = current["weathercode"]

            condition = weather_codes.get(code, "알 수 없음")

            return f"상태: {condition}\n온도: {temp}°C\n풍속: {wind} km/h"

    except Exception as e:
        return f"요청 실패: {e}"

In [20]:
get_weather(37.500078, 127.035548)

'상태: 흐림 ☁️\n온도: 22.2°C\n풍속: 4.9 km/h'

## 함수 -> 툴로 변경 후 제공

In [21]:
class CoordinatesToolArgSchema(BaseModel):
    location_name: str = Field("위도와 경도로 바꾸고 싶은 장소명입니다.")

class CoordinatesTool(BaseTool):
    name: Type[str] = "coordinates_tool"
    description: Type[str] = """
        장소명을 위도(latitude)와 경도(longitude) 좌표로 변환합니다.
        장소명을 위도와 경도로 변환하고 싶을 대 사용하는 도구입니다.
    """
    args_schema: Type[BaseModel] = CoordinatesToolArgSchema

    def _run(self, location_name: str) -> Tuple[float, float]:
        locator = Nominatim(user_agent="ksh")
        location = locator.geocode(location_name)
    
        return location.latitude, location.longitude, 

In [22]:
class WeatherSearchToolArgSchema(BaseModel):
    lat: float = Field(description="위도, Example Value: 37.500078")
    lon: float = Field(description="경도, Example Value: 127.035548")
    
class WeatherSearchTool(BaseTool):
    name: Type[str] = "weather_search_tool"
    description: Type[str] = """
        지역의 날씨를 가져오고 싶을 때 사용하는 툴입니다.
        위도와 경도를 입력하면, 해당 지역의 날씨의 정보를 문자열로 반환합니다.
    """

    args_schema: Type[BaseModel] = WeatherSearchToolArgSchema
    
    # 위도, 경도 -> 날씨
    def _run(self, lat: float, lon: float) -> str:
        url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        
        # 기상 코드(WMO Code)를 한글로 변환하는 딕셔너리
        weather_codes = {
            0: "맑음 ☀️",
            1: "대체로 맑음 🌤️", 2: "구름 조금 ⛅", 3: "흐림 ☁️",
            45: "안개 🌫️", 48: "침강 안개 🌫️",
            51: "가벼운 이슬비 🌦️", 53: "이슬비 🌧️", 55: "강한 이슬비 ⛈️",
            61: "약한 비 💧", 63: "보통 비 ☔", 65: "강한 비 🌊",
            71: "약한 눈 ❄️", 73: "보통 눈 ☃️", 75: "강한 눈 🏔️",
            80: "약한 소나기 🌦️", 81: "보통 소나기 🌧️", 82: "강한 소나기 ⛈️",
            95: "뇌우 ⚡", 96: "뇌우 및 우박 ⛈️", 99: "심한 뇌우 🌪️"
        }
    
        try:
            response = requests.get(url)
            datas = response.json()
    
            if "current_weather" in datas:
                current = datas["current_weather"]
                temp = current["temperature"]
                wind = current["windspeed"]
                code = current["weathercode"]
    
                condition = weather_codes.get(code, "알 수 없음")
    
                return f"상태: {condition}\n온도: {temp}°C\n풍속: {wind}km/h"
            
        except Exception as e:
            return "요청 실패"

In [23]:
tools = []

agent = create_agent(
    model="gpt-4o-mini",
    tools=tools
)

prompt = ChatPromptTemplate.from_messages([
    ("human", "강남의 실시간 날씨 정보를 알려줘 !")
])

chain = prompt | agent
result = chain.invoke({})

result

{'messages': [HumanMessage(content='강남의 실시간 날씨 정보를 알려줘 !', additional_kwargs={}, response_metadata={}, id='4b78925c-1bff-411c-ac4e-e87e78630e88'),
  AIMessage(content='죄송하지만, 실시간 날씨 정보를 제공할 수는 없습니다. 하지만 강남의 날씨를 확인하고 싶으시다면, 기상청 웹사이트나 날씨 관련 앱을 이용하시면 정확한 정보를 얻으실 수 있습니다. 도움이 필요하시면 언제든지 말씀해 주세요!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 18, 'total_tokens': 81, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_6ade39f7ff', 'id': 'chatcmpl-Dh8WjVR7Wv09Q7UEHhDO86IIBJBfz', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e3efd-6283-7210-ba07-27141d00cdc5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_t

In [24]:
prompt = ChatPromptTemplate.from_messages([
    ("human", "엔비디아 사도 돼 ?")
])

chain = prompt | agent
result = chain.invoke({})

result

{'messages': [HumanMessage(content='엔비디아 사도 돼 ?', additional_kwargs={}, response_metadata={}, id='555f3686-a53c-4662-96c0-336a08de462a'),
  AIMessage(content='엔비디아(NVIDIA) 주식에 투자할지 여부는 여러 가지 요소에 따라 달라질 수 있습니다. 주식 투자 결정을 내리기 전에 고려해야 할 몇 가지 사항은 다음과 같습니다.\n\n1. **기업 실적**: 엔비디아의 최근 실적 보고서를 검토하고 수익, 성장률, 마진 등을 확인하세요.\n\n2. **업종 전망**: 엔비디아는 인공지능(AI), 데이터 센터, 게임 등 다양한 분야에서 활동하고 있습니다. 이들 산업의 전망과 시장의 트렌드를 분석해 보세요.\n\n3. **경쟁력**: 엔비디아의 경쟁사 및 시장에서의 위치를 비교해 보세요. 경쟁사가 어떻게 변화하고 있는지, 엔비디아의 차별화된 점은 무엇인지 확인하는 것이 중요합니다.\n\n4. **주가 변동성**: 주가의 역사적 변동성을 분석하여 투자에 따른 위험 요소를 고려하세요.\n\n5. **투자 목표**: 당신의 투자 목표와 시간 프레임이 무엇인지 명확히 하세요. 단기 투자인지 장기 투자인지에 따라 전략이 달라질 수 있습니다.\n\n6. **전문가 의견**: 금융 전문가의 분석 및 의견을 참고하는 것도 유익할 수 있습니다. 하지만 최종 결정은 본인이 해야 합니다.\n\n결국, 투자 결정은 개인의 판단에 기반해야 하며, 충분한 정보를 수집하고 분석한 후 결정하는 것이 중요합니다. 전문가와 상담하는 것도 좋은 방법입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 327, 'prompt_tokens': 16, 'total_tokens': 343, 'completion_tokens_details': {'accepted_p

1. DuckDuckgo search tool
   - 회사 정보를 웹에서 찾는 툴을 만들기
   - 회사가 상장했는가, 회사 주식 심볼(ticker, 티커)는 무엇인지?
   - 가령 A라는 회사에 대한 정보를 찾고자 한다면, agent에 의해 툴은 A가 어떤 회사인지 검색을 수행하게 한다

2. AlphaVantaga API(주식 회사 정보)
   - 1) 회사의 심볼을 알아내는 툴
   - 2) 손익 계산서를 위한 툴
   - 3) 뉴스 심리지수를 위한 툴
   - 4) 회사의 개요를 위한 툴

   - https://www.alphavantage.co/
   - 위 사이트에 접속 후 API_KEY 발급
   - 환경변수에 등록하기
   - ALPHA_VANTAGE_API_KEY="발급받은 키"

In [36]:
from dotenv import load_dotenv
import os
import requests

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain.tools import tool, BaseTool
from langchain.agents import create_agent
from langchain_openai.chat_models.base import ChatOpenAI

from pydantic import BaseModel, Field
from typing import Any, Type, List, Tuple, Dict #Python의 내장 모듈 typing

# 추가된 import
from langchain_community.utilities.duckduckgo_search import DuckDuckGoSearchAPIWrapper
from geopy.geocoders import Nominatim

load_dotenv()
print(os.environ.get('OPENAI_API_KEY')[:20])

sk-proj-ktAbTJ1mXZ2V


## 1. 일 주 월 단위의 실적을 제공
    - https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol=IBM&apikey=demo

    

In [27]:
"""
    Meta Data": {
    "1. Information": "Daily Prices (open, high, low, close) and Volumes",
    "2. Symbol": "IBM", #회사의 티커 
    "3. Last Refreshed": "2026-05-18",
    "4. Output Size": "Compact",
    "5. Time Zone": "US/Eastern"
    },
    "Time Series (Daily)": {
    "2026-05-18": {
    "1. open": "218.5500", #시작가
    "2. high": "223.3300", #고점
    "3. low": "217.7500", #저점
    "4. close": "222.7500", # 종가
    "5. volume": "5946367" #거래량
    },
    "2026-05-15": {
    "1. open": "218.2000",
    "2. high": "220.9100",
    "3. low": "217.6150",
    "4. close": "219.3000",
    "5. volume": "6154450"
    },
"""

'\n    Meta Data": {\n    "1. Information": "Daily Prices (open, high, low, close) and Volumes",\n    "2. Symbol": "IBM", #회사의 티커 \n    "3. Last Refreshed": "2026-05-18",\n    "4. Output Size": "Compact",\n    "5. Time Zone": "US/Eastern"\n    },\n    "Time Series (Daily)": {\n    "2026-05-18": {\n    "1. open": "218.5500", #시작가\n    "2. high": "223.3300", #고점\n    "3. low": "217.7500", #저점\n    "4. close": "222.7500", # 종가\n    "5. volume": "5946367" #거래량\n    },\n    "2026-05-15": {\n    "1. open": "218.2000",\n    "2. high": "220.9100",\n    "3. low": "217.6150",\n    "4. close": "219.3000",\n    "5. volume": "6154450"\n    },\n'

## 2. News & Sentiments (최신뉴스 & 민감도)
- https://www.alphavantage.co/query?function=NEWS_SENTIMENT&tickers=AAPL&apikey=demo

In [29]:
"""
    "items": "50",
    # 뉴스의 지표
    "sentiment_score_definition": "x <= -0.35: Bearish; -0.35 < x <= -0.15: Somewhat-Bearish; -0.15 < x < 0.15: Neutral; 0.15 <= x < 0.35: Somewhat_Bullish; x >= 0.35: Bullish",
    "relevance_score_definition": "0 < x <= 1, with a higher score indicating higher relevance.",
    "feed": [
    {
    "title": "Apple CEO Tim Cook in Beijing with US Presidential Delegation – May 2026 - News and Statistics",
    "url": "https://www.indexbox.io/blog/tim-cook-joins-us-delegation-to-beijing-as-apple-navigates-china-ties/",
    "time_published": "20260519T031958",
    "authors": [],
    "summary": "Apple CEO Tim Cook is in Beijing as part of a U.S. presidential delegation, marking the first such visit in nearly a decade. This trip is crucial for Apple due to its extensive manufacturing operations in China and China being its largest market outside the U.S. Improved U.S.-China relations, including reduced tariffs and increased market access, would significantly benefit Apple, especially following Chinese President Xi Jinping's recent pledge to \"open wider\" for American businesses.",
    "banner_image": "https://www.indexbox.io/landing/img/blog/telegram-fallback/5eab958188a1a3d707e92b07af013cb6.webp",
    "source": "IndexBox",
    "category_within_source": "General",
    "source_domain": "IndexBox",
    "topics": [
    {
    "topic": "technology",
    "relevance_score": "0.812212"
    },
    {
    "topic": "economy_macro",
    "relevance_score": "0.745762"
    },
    {
    "topic": "finance",
    "relevance_score": "0.634877"
    },
    {
    "topic": "manufacturing",
    "relevance_score": "0.647957"
    }
    ],
"""

'\n    "items": "50",\n    # 뉴스의 지표\n    "sentiment_score_definition": "x <= -0.35: Bearish; -0.35 < x <= -0.15: Somewhat-Bearish; -0.15 < x < 0.15: Neutral; 0.15 <= x < 0.35: Somewhat_Bullish; x >= 0.35: Bullish",\n    "relevance_score_definition": "0 < x <= 1, with a higher score indicating higher relevance.",\n    "feed": [\n    {\n    "title": "Apple CEO Tim Cook in Beijing with US Presidential Delegation – May 2026 - News and Statistics",\n    "url": "https://www.indexbox.io/blog/tim-cook-joins-us-delegation-to-beijing-as-apple-navigates-china-ties/",\n    "time_published": "20260519T031958",\n    "authors": [],\n    "summary": "Apple CEO Tim Cook is in Beijing as part of a U.S. presidential delegation, marking the first such visit in nearly a decade. This trip is crucial for Apple due to its extensive manufacturing operations in China and China being its largest market outside the U.S. Improved U.S.-China relations, including reduced tariffs and increased market access, would sig

## 3. 회사 재무재표 (손익)
-https://www.alphavantage.co/query?function=INCOME_STATEMENT&symbol=IBM&apikey=demo

In [30]:
"""
        {
        "symbol": "IBM",
        "annualReports": [
        {
        "fiscalDateEnding": "2025-12-31",
        "reportedCurrency": "USD",
        "grossProfit": "40185000000", # 순수익
        "totalRevenue": "67535000000", #총매출
        "costOfRevenue": "27350000000", #매출원가
        "costofGoodsAndServicesSold": "27350000000",
        "operatingIncome": "10325000000",
        "sellingGeneralAndAdministrative": "18285000000",
        "researchAndDevelopment": "8320000000",
        "operatingExpenses": "29860000000",
        "investmentIncomeNet": "None",
        "netInterestIncome": "-1290000000",
        "interestIncome": "645000000",
        "interestExpense": "1935000000",
        "nonInterestIncome": "None",
        "otherNonOperatingIncome": "None",
        "depreciation": "None",
        "depreciationAndAmortization": "5021000000",
        "incomeBeforeTax": "10328000000",
        "incomeTaxExpense": "-242000000",
        "interestAndDebtExpense": "None",
        "netIncomeFromContinuingOperations": "10571000000",
        "comprehensiveIncomeNetOfTax": "None",
        "ebit": "12263000000",
        "ebitda": "17284000000",
        "netIncome": "10593000000" # 영업이익
        },
    
"""

'\n        {\n        "symbol": "IBM",\n        "annualReports": [\n        {\n        "fiscalDateEnding": "2025-12-31",\n        "reportedCurrency": "USD",\n        "grossProfit": "40185000000", # 순수익\n        "totalRevenue": "67535000000", #총매출\n        "costOfRevenue": "27350000000", #매출원가\n        "costofGoodsAndServicesSold": "27350000000",\n        "operatingIncome": "10325000000",\n        "sellingGeneralAndAdministrative": "18285000000",\n        "researchAndDevelopment": "8320000000",\n        "operatingExpenses": "29860000000",\n        "investmentIncomeNet": "None",\n        "netInterestIncome": "-1290000000",\n        "interestIncome": "645000000",\n        "interestExpense": "1935000000",\n        "nonInterestIncome": "None",\n        "otherNonOperatingIncome": "None",\n        "depreciation": "None",\n        "depreciationAndAmortization": "5021000000",\n        "incomeBeforeTax": "10328000000",\n        "incomeTaxExpense": "-242000000",\n        "interestAndDebtExpense":

## 4. 회사의 개요 (Company OverView)
- https://www.alphavantage.co/query?function=OVERVIEW&symbol=IBM&apikey=demo

In [31]:
"""
    {
    "Symbol": "IBM",
    "AssetType": "Common Stock",
    "Name": "International Business Machines",
    "Description": "International Business Machines Corporation (IBM) is an American multinational technology company headquartered in Armonk, New York, with operations in over 170 countries. The company began in 1911, founded in Endicott, New York, as the Computing-Tabulating-Recording Company (CTR) and was renamed International Business Machines in 1924. IBM is incorporated in New York. IBM produces and sells computer hardware, middleware and software, and provides hosting and consulting services in areas ranging from mainframe computers to nanotechnology. IBM is also a major research organization, holding the record for most annual U.S. patents generated by a business (as of 2020) for 28 consecutive years. Inventions by IBM include the automated teller machine (ATM), the floppy disk, the hard disk drive, the magnetic stripe card, the relational database, the SQL programming language, the UPC barcode, and dynamic random-access memory (DRAM). The IBM mainframe, exemplified by the System/360, was the dominant computing platform during the 1960s and 1970s.",
    "CIK": "51143",
    "Exchange": "NYSE",
    "Currency": "USD",
    "Country": "USA",
    "Sector": "TECHNOLOGY",
    "Industry": "INFORMATION TECHNOLOGY SERVICES",
    "Address": "ONE NEW ORCHARD ROAD, ARMONK, NY, UNITED STATES, 10504",
    "OfficialSite": "https://www.ibm.com",
    "FiscalYearEnd": "December",
    "LatestQuarter": "2026-03-31",
    "MarketCapitalization": "206116848000",
    "EBITDA": "16611000000",
    "PERatio": "19.42",
    "PEGRatio": "2.152",
    "BookValue": "35.08",
    "DividendPerShare": "6.72",
    "DividendYield": "0.0308",
    "EPS": "11.29",
    "RevenuePerShareTTM": "73.71",
    "ProfitMargin": "0.156",
    "OperatingMarginTTM": "0.138",
    "ReturnOnAssetsTTM": "0.0537",
    "ReturnOnEquityTTM": "0.358",
    "RevenueTTM": "68910998000",
    "GrossProfitTTM": "40214999000",
    "DilutedEPSTTM": "11.29",
    "QuarterlyEarningsGrowthYOY": "0.142",
    "QuarterlyRevenueGrowthYOY": "0.095",
    "AnalystTargetPrice": "278.18",
    "AnalystRatingStrongBuy": "1",
    "AnalystRatingBuy": "10",
    "AnalystRatingHold": "9",
    "AnalystRatingSell": "0",
    "AnalystRatingStrongSell": "1",
    "TrailingPE": "19.42",
    "ForwardPE": "18.62",
    "PriceToSalesRatioTTM": "2.991",
    "PriceToBookRatio": "6.55",
    "EVToRevenue": "3.976",
    "EVToEBITDA": "15.54",
    "Beta": "0.581",
    "52WeekHigh": "320.7",
    "52WeekLow": "212.34",
    "50DayMovingAverage": "239.57",
    "200DayMovingAverage": "270.38",
    "SharesOutstanding": "939885000",
    "SharesFloat": "937902000",
    "PercentInsiders": "0.117",
    "PercentInstitutions": "65.558",
    "DividendDate": "2026-06-10",
    "ExDividendDate": "2026-05-08"
    }


"""

'\n    {\n    "Symbol": "IBM",\n    "AssetType": "Common Stock",\n    "Name": "International Business Machines",\n    "Description": "International Business Machines Corporation (IBM) is an American multinational technology company headquartered in Armonk, New York, with operations in over 170 countries. The company began in 1911, founded in Endicott, New York, as the Computing-Tabulating-Recording Company (CTR) and was renamed International Business Machines in 1924. IBM is incorporated in New York. IBM produces and sells computer hardware, middleware and software, and provides hosting and consulting services in areas ranging from mainframe computers to nanotechnology. IBM is also a major research organization, holding the record for most annual U.S. patents generated by a business (as of 2020) for 28 consecutive years. Inventions by IBM include the automated teller machine (ATM), the floppy disk, the hard disk drive, the magnetic stripe card, the relational database, the SQL programm

## Stock Tools 생성

In [53]:
class StockMartSymbolSearchToolArgSchema(BaseModel):
    query: str = Field(description="""
        The query you will search for Example query: Stock Market Symbol for Apple company.
    """)

class StockMartSymbolSearchTool(BaseTool):
    name: Type[str] = "stock_mark_symbol_search_tool"
    description: Type[str] = """
        Use this tool fin the stock market symbol for a company.
        It takes a query as an argument.
    """

    args_schema: Type[BaseModel] = StockMartSymbolSearchToolArgSchema
    
    def _run(self, query):
        ddg = DuckDuckGoSearchAPIWrapper()
        ddg.run(query)

In [49]:
ddg = DuckDuckGoSearchAPIWrapper()
ddg.run("apple stock market symbol")

'... for Apple ’ s stock using its ticker symbol AAPL. You can then place an order to buy shares of Apple stock at the current market price. Investing in individual stocks like Apple can be more dangerous than investing in ETFs or mutual ... The symbol of Apple ’ s stock is AAPL. A stock symbol is a unique code, abbreviation, or a unique combination of letters assigned to a ... For example, the stock symbol for Apple Inc. A stock symbol is a unique code, abbreviation, or a unique combination of letters assigned to a ... For example, the stock symbol for Apple Inc. So, if you want to buy or sell Apple Inc share, you search for its stock symbol on your broker ’ s platform to find out its latest buy and sell ...'

In [50]:
def parse_output(result) :
    return result["messages"][-1].content

In [52]:
tools = [StockMarketSymbolSearchTool()]

agent = create_agent(
    model="gpt-4o-mini",
    tools=tools
)

prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}")
])

chain = prompt | agent | RunnableLambda(parse_output)

result = chain.invoke({
    "question" : "엔비디아의 심볼과 회사의 개요, 회사의 손익계산서, 뉴스등을 고려해서 엔비디아를 구매해야하는지 알려줘"
})

C:\back_0900_kyj\python\workspace\venv\Lib\site-packages\pydantic\json_schema.py:2466: PydanticJsonSchemaWarning: Default value <class '__main__.StockMarketSymbolSearchTool'> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
C:\back_0900_kyj\python\workspace\venv\Lib\site-packages\pydantic\json_schema.py:2466: PydanticJsonSchemaWarning: Default value <class '__main__.StockMarketSymbolSearchTool'> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
C:\back_0900_kyj\python\workspace\venv\Lib\site-packages\pydantic\json_schema.py:2466: PydanticJsonSchemaWarning: Default value <class '__main__.StockMarketSymbolSearchTool'> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
C:\back_0900_kyj\python\workspace\venv\Lib\site-package

In [55]:
print(result)

엔비디아의 주식 심볼을 찾는 데 문제가 발생했습니다. 하지만, 엔비디아(NVIDIA)는 미국 증권거래소에서 "NVDA"라는 심볼로 거래됩니다.

이제 엔비디아에 대한 회사 개요, 손익계산서, 최근 뉴스 등을 제공하기 위해 다른 리소스를 참조하거나 구체적인 데이터를 요청할 수 있습니다. 어떤 정보를 원하시는지 추가로 말씀해 주시면 더 도와드리겠습니다.


In [ ]:
alpha_vantage_api_key = os.environ.get("ALPHA_VANTAGE_API_KEY")

## News Sentiments Tool

In [62]:
class NewsSentimentsToolArgsSchema(BaseModel):
    symbol: str = Field(description = "Stock symbol of the company. Example : AAPL , TSLA")

class NewsSentimentsTool(BaseTool):
    name : Type[str] = "news_sentiment_tool"
    description: Type[str] = """
    Use this to get the news sentiment of a company.
    tou should enter a stock symbol.
    """

    args_schema : Type[BaseModel] = NewsSentimentsToolArgsSchema
    
    def _run(self, symbol):
        url = f"https://www.alphavantage.co/query?function=NEWS_SENTIMENT&tickers={symbol}&apikey={alpha_vantage_api_key}"
        response = request.get(url)
        datas = response.json()
        return datas

In [64]:
tools = [
    StockMarketSymbolSearchTool(), #회사 심볼 툴
    NewsSentimentsTool() # 회사 뉴스,민감도 툴
]

agent = create_agent(
    model="gpt-4o-mini",
    tools=tools
)

prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}")
])

chain = prompt | agent | RunnableLambda(parse_output)

result = chain.invoke({
    "question" : "엔비디아의 심볼과 회사의 개요, 회사의 손익계산서, 뉴스등을 고려해서 엔비디아를 구매해야하는지 알려줘"
})

C:\back_0900_kyj\python\workspace\venv\Lib\site-packages\pydantic\json_schema.py:2466: PydanticJsonSchemaWarning: Default value <class '__main__.StockMarketSymbolSearchTool'> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
C:\back_0900_kyj\python\workspace\venv\Lib\site-packages\pydantic\json_schema.py:2466: PydanticJsonSchemaWarning: Default value <class '__main__.StockMarketSymbolSearchTool'> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
C:\back_0900_kyj\python\workspace\venv\Lib\site-packages\pydantic\json_schema.py:2466: PydanticJsonSchemaWarning: Default value <class '__main__.StockMarketSymbolSearchTool'> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
C:\back_0900_kyj\python\workspace\venv\Lib\site-package

TypeError: StockMarketSymbolSearchTool._run() got an unexpected keyword argument 'name'

In [ ]:
print (result)

In [65]:
## 회사의 재무재표 ,손익 툴

In [71]:
class CompanyIncomeStatementToolArgsSchema(BaseModel):
    symbol: str = Field(description="Stock symbol of the company. Exmaple: AAPL, TSLA")

class CompanyIncomeStatementTool(BaseTool):
    name: Type[str] = "company_income_statement_tool"
    description: Type[str] = """
        Use this to get the income statement of a company.
        You should enter a stock symbol
    """

    args_schema: Type[BaseModel] = CompanyIncomeStatementToolArgsSchema
    
    def _run(self, symbol: str) -> Dict[str, any]:
        url = f"https://www.alphavantage.co/query?function=INCOME_STATEMENT&symbol={symbol}&apikey={alpha_vantage_api_kay}"
        response = requests.get(url)
        datas = response.json()
        return datas

In [74]:
tools = [
    StockMarketSymbolSearchTool(), #회사 심볼 툴
    NewsSentimentsTool(), # 회사 뉴스,민감도 툴
    CompanyIncomeStatementTool(),
]

agent = create_agent(
    model="gpt-4o-mini",
    tools=tools
)

prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}")
])

chain = prompt | agent | RunnableLambda(parse_output)

result = chain.invoke({
    "question" : "엔비디아의 심볼과 회사의 개요, 회사의 손익계산서, 뉴스등을 고려해서 엔비디아를 구매해야하는지 알려줘"
})

C:\back_0900_kyj\python\workspace\venv\Lib\site-packages\pydantic\json_schema.py:2466: PydanticJsonSchemaWarning: Default value <class '__main__.StockMarketSymbolSearchTool'> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
C:\back_0900_kyj\python\workspace\venv\Lib\site-packages\pydantic\json_schema.py:2466: PydanticJsonSchemaWarning: Default value <class '__main__.StockMarketSymbolSearchTool'> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
C:\back_0900_kyj\python\workspace\venv\Lib\site-packages\pydantic\json_schema.py:2466: PydanticJsonSchemaWarning: Default value <class '__main__.StockMarketSymbolSearchTool'> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
C:\back_0900_kyj\python\workspace\venv\Lib\site-package

GraphRecursionError: Recursion limit of 25 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/GRAPH_RECURSION_LIMIT

In [73]:
print(result)

엔비디아의 주식 심볼을 찾는 데 문제가 발생했습니다. 하지만, 엔비디아(NVIDIA)는 미국 증권거래소에서 "NVDA"라는 심볼로 거래됩니다.

이제 엔비디아에 대한 회사 개요, 손익계산서, 최근 뉴스 등을 제공하기 위해 다른 리소스를 참조하거나 구체적인 데이터를 요청할 수 있습니다. 어떤 정보를 원하시는지 추가로 말씀해 주시면 더 도와드리겠습니다.


## 4. 회사 개요 툴 , 주가정보 툴

In [76]:
class CompanyOverviewToolArgsSchema(BaseModel):
    symbol: str = Field(description="Stock symbol of the company. Exmaple: AAPL, TSLA")

class CompanyOverviewTool(BaseTool):
    name: Type[str] = "company_overview_Tool"
    description: Type[str] = """
        Use this to get the overview of a company.
        You should enter a stock symbol
    """

    args_schema: Type[BaseModel] = CompanyOverviewToolArgsSchema
    
    def _run(self, symbol: str) -> Dict[str, any]:
        url = f"https://www.alphavantage.co/query?function=OVERVIEW&symbol={symbol}&apikey={alpha_vantage_api_kay}"
        response = requests.get(url)
        datas = response.json()
        return datas


class CompanyStockPerformanceToolArgsSchema(BaseModel):
    symbol: str = Field(description="Stock symbol of the company. Exmaple: AAPL, TSLA")

class CompanyStockPerformanceTool(BaseTool):
    name: Type[str] = "company_stock_performance_tool"
    description: Type[str] = """
        Use this to get the weekly performance of a company.
        You should enter a stock symbol
    """

    args_schema: Type[BaseModel] = CompanyStockPerformanceToolArgsSchema
    
    def _run(self, symbol: str) -> Dict[str, any]:
        url = f"https://www.alphavantage.co/query?function=TIME_SERIES_WEEKLY&symbol={symbol}&apikey={alpha_vantage_api_kay}"
        response = requests.get(url)
        datas = response.json()
        return datas

In [77]:
prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}")
])

chain = prompt | agent | RunnableLambda(parse_output)

result = chain.invoke({
    "question" : "엔비디아의 심볼과 회사의 개요, 회사의 손익계산서, 뉴스등을 고려해서 엔비디아를 구매해야하는지 알려줘"
})

C:\back_0900_kyj\python\workspace\venv\Lib\site-packages\pydantic\json_schema.py:2466: PydanticJsonSchemaWarning: Default value <class '__main__.StockMarketSymbolSearchTool'> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
C:\back_0900_kyj\python\workspace\venv\Lib\site-packages\pydantic\json_schema.py:2466: PydanticJsonSchemaWarning: Default value <class '__main__.StockMarketSymbolSearchTool'> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
C:\back_0900_kyj\python\workspace\venv\Lib\site-packages\pydantic\json_schema.py:2466: PydanticJsonSchemaWarning: Default value <class '__main__.StockMarketSymbolSearchTool'> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


NameError: name 'alpha_vantage_api_kay' is not defined

In [ ]:
result = chain.invoke({
    "company" : "엔비디아"
})

In [ ]:
print(result)

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """
        You are a veteran Wall Street stock investment expert and a cold-blooded Chief Financial Analyst. 
        Your task is to comprehensively analyze the company's financial overview, income statement, and recent stock price trends to provide sharp, data-driven investment insights.

        [CORE DIRECTIVES]
        1. Avoid ambiguity. Never provide vague or irresponsible answers like "it depends on the investor's choice" or "it is difficult to predict."
        2. Make a definitive call. Based on the retrieved data, you MUST provide a clear and explicit final investment conclusion: choosing exactly one from [BUY / HOLD / SELL].
        3. Maintain a highly professional, objective, and authoritative tone. Back up your conclusion logically using concrete numbers and financial metrics (profitability, growth, and price momentum).
        4. Language Requirement: You MUST write the final answer entirely in Korean. Even though the analysis is based on English data, the final output delivered to the user must be in clear, professional Korean.
        
        Your analysis will guide critical financial decisions. Be ruthless, objective, and strictly rely on the data provided.
    """),
    ("human", """
        You must use tools to answer this question.
    
        1. Find the stock symbol for {company}.
        2. Retrieve the company's financial overview.
        3. Retrieve the company's income statement.
        4. Retrieve the stock price data (recent price, trend, or performance).
        5. Retrieve at least 5 recent news articles for {company} along with their sources (publisher or URL), and analyze the overall news sentiment.
        6. Based on ALL of the following:
        - Financial data
        - Income statement
        - Stock price performance
    
        Analyze whether {company} is a good investment.
    
        Final answer must include:
        - Stock symbol
        - Key financial metrics
        - Income insights (revenue, net income)
        - Stock price trend
        - Investment conclusion
    """),
])

chain = prompt | agent | RunnableLambda(parse_output)